In [ ]:
!pip install pymupdf
!pip install pandas

   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ------- -------------------------------- 1.8/10.0 MB 9.9 MB/s eta 0:00:01
   ----------- ---------------------------- 2.9/10.0 MB 7.1 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/10.0 MB 6.3 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/10.0 MB 5.6 MB/s eta 0:00:02
   ---------------- ----------------------- 4.2/10.0 MB 4.7 MB/s eta 0:00:02
   ----------------- ---------------------- 4.5/10.0 MB 4.1 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/10.0 MB 3.4 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/10.0 MB 3.3 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/10.0 MB 3.1 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/10.0 MB 3.0 MB/s eta 0:00:02
   -------------------------- ------------- 6.6/10.0 MB 2.9 MB/s eta 0:00:02
   --------------------------- ------------ 6.8/10.0 MB 2.8 MB/s eta 0:00:02
   ---

Could not find platform independent libraries <prefix>


In [15]:
import os
import csv
import json
import fitz
import glob
from collections import defaultdict



In [3]:
import fitz

doc = fitz.open(r"ranklist\pdf\2025.pdf")
total_pages = len(doc)
print(total_pages)


5439


In [4]:
import math
import fitz

input_pdf = r"ranklist\pdf\2025.pdf"
doc = fitz.open(input_pdf)
total_pages = len(doc)
num_parts = 4

chunk_size = math.ceil(total_pages / num_parts)

for i in range(num_parts):
    start_page = i * chunk_size
    end_page = min((i + 1) * chunk_size - 1, total_pages - 1)

    if start_page >= total_pages:
        break

    part_doc = fitz.open()
    part_doc.insert_pdf(doc, from_page=start_page, to_page=end_page)

    output_filename = f"ranklist\\pdf\\2025-p{i+1}.pdf"
    part_doc.save(output_filename)
    part_doc.close()

    print(
        f"Saved {output_filename}: pages {start_page + 1} to {end_page + 1} ({end_page - start_page + 1} pages)"
    )

doc.close()


Saved ranklist\pdf\2025-p1.pdf: pages 1 to 1360 (1360 pages)
Saved ranklist\pdf\2025-p2.pdf: pages 1361 to 2720 (1360 pages)
Saved ranklist\pdf\2025-p3.pdf: pages 2721 to 4080 (1360 pages)
Saved ranklist\pdf\2025-p4.pdf: pages 4081 to 5439 (1359 pages)


In [9]:
!node "c:\Users\theve\OneDrive\Documents\pdf-table-extractor\parse-cmd.js" ranklist\pdf\2026.pdf > x.json


^C


In [2]:
print(os.getcwd())

c:\Users\theve\Downloads\rankra\etl


In [2]:
fi = open("ranklist/2025.json","r")

In [ ]:
fi.seek(0)  # Move file pointer back to the start
d = json.load(fi)


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

: 

In [11]:
def rank_csv(year: int):
    rows = []
    files = sorted(glob.glob(f"ranklist/{year}-p*.json"))
    
    if not files:
        single_file = f"ranklist/{year}.json"
        if os.path.exists(single_file):
            files = [single_file]
    for filepath in files:
        with open(filepath, "r", encoding="utf-8") as fi:
            d = json.load(fi)
        
        for page in d.get("pageTables", []):
            for row in page.get("tables", []):
                try:
                    cutoff = float(row[2].strip())
                    general_rank = int(row[3].strip())
                    community = row[4].strip()
                    community_rank = int(row[5].strip()) if len(row) > 5 and row[5].strip().isdigit() else row[5].strip() if len(row) > 5 else ""
                    
                    rows.append({
                        "general_rank": general_rank,
                        "cutoff": cutoff,
                        "community": community,
                        "community_rank": community_rank
                    })
                except (ValueError, IndexError):
                    continue
                    
    with open(f"ranklist/{year}.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["general_rank", "cutoff", "community", "community_rank"])
        writer.writeheader()
        writer.writerows(rows)

rank_csv(2025)

In [13]:
def check_linear(csv_path: str):
    is_linear = True
    missing_ranks = []
    duplicate_ranks = []
    out_of_order = []
    
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        expected_rank = 1
        seen_ranks = set()
        
        for row_num, row in enumerate(reader, start=1):
            try:
                rank = int(row["general_rank"])
            except (ValueError, KeyError):
                continue
                
            if rank in seen_ranks:
                duplicate_ranks.append((row_num, rank))
            seen_ranks.add(rank)
            
            if rank != expected_rank:
                is_linear = False
                if rank > expected_rank:
                    missing_ranks.extend(range(expected_rank, rank))
                else:
                    out_of_order.append((row_num, rank, expected_rank))
                    
            expected_rank = rank + 1
    print("Strictly Linear:", is_linear)
    print("Total Ranks Processed:", len(seen_ranks))
    
    if duplicate_ranks:
        print(f"Duplicates ({len(duplicate_ranks)}):", duplicate_ranks[:5])
    if missing_ranks:
        print(f"Missing Ranks ({len(missing_ranks)}):", missing_ranks[:10])
    if out_of_order:
        print(f"Out of Order ({len(out_of_order)}):", out_of_order[:5])
check_linear(r"ranklist\2025.csv")
check_linear(r"ranklist\2026.csv")

Strictly Linear: True
Total Ranks Processed: 239299
Strictly Linear: True
Total Ranks Processed: 233812


In [17]:
def load_ranklist(filepath):
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["community_rank"].isdigit():
                data.append({
                    "general_rank": int(row["general_rank"]),
                    "cutoff": float(row["cutoff"]),
                    "community": row["community"],
                    "community_rank": int(row["community_rank"])
                })
    return data
d25 = load_ranklist(r"ranklist\2025.csv")
d26 = load_ranklist(r"ranklist\2026.csv")
def analyze_community_shift(d25, d26):
    m25 = defaultdict(dict)
    m26 = defaultdict(dict)
    
    for r in d25:
        m25[r["community"]][r["community_rank"]] = (r["general_rank"], r["cutoff"])
        
    for r in d26:
        m26[r["community"]][r["community_rank"]] = (r["general_rank"], r["cutoff"])
        
    check_ranks = [500, 1000, 2500, 5000, 10000, 20000, 40000]
    communities = ["BC", "BCM", "MBC", "SC", "SCA", "ST"]
    
    for comm in communities:
        print(f"\n==================== COMMUNITY: {comm} ====================")
        print(f"{'Comm Rank':<10} | {'2025 Gen':<10} | {'2026 Gen':<10} | {'2025 Ratio':<10} | {'2026 Ratio':<10} | {'Shift Direction':<15}")
        print("-" * 78)
        
        for cr in check_ranks:
            if cr in m25[comm] and cr in m26[comm]:
                g25, c25 = m25[comm][cr]
                g26, c26 = m26[comm][cr]
                
                r25 = g25 / cr
                r26 = g26 / cr
                
                diff = g26 - g25
                if diff < 0:
                    status = f"FORWARD ({diff})"
                elif diff > 0:
                    status = f"BACKWARD (+{diff})"
                else:
                    status = "SAME"
                    
                print(f"{cr:<10} | {g25:<10} | {g26:<10} | {r25:<10.2f} | {r26:<10.2f} | {status:<15}")
analyze_community_shift(d25, d26)


==================== COMMUNITY: BC ====================
Comm Rank  | 2025 Gen   | 2026 Gen   | 2025 Ratio | 2026 Ratio | Shift Direction
------------------------------------------------------------------------------
500        | 980        | 824        | 1.96       | 1.65       | FORWARD (-156) 
1000       | 1875       | 1669       | 1.88       | 1.67       | FORWARD (-206) 
2500       | 4590       | 4315       | 1.84       | 1.73       | FORWARD (-275) 
5000       | 9091       | 8573       | 1.82       | 1.71       | FORWARD (-518) 
10000      | 18228      | 17360      | 1.82       | 1.74       | FORWARD (-868) 
20000      | 36914      | 35738      | 1.85       | 1.79       | FORWARD (-1176)
40000      | 76970      | 75631      | 1.92       | 1.89       | FORWARD (-1339)

==================== COMMUNITY: BCM ====================
Comm Rank  | 2025 Gen   | 2026 Gen   | 2025 Ratio | 2026 Ratio | Shift Direction
-----------------------------------------------------------------------------